# Implementation of Semidefinite Programming

## Using CVXPY

### Basic Example

In [8]:
import cvxpy as cp
import numpy as np

np.random.seed(42)

n = 10

C_raw = np.random.randn(n, n)
C = C_raw.T @ C_raw  # PSD by construction

A = [np.random.randn(n, n) for _ in range(5)]
A = [(a + a.T) / 2 for a in A]
b = np.random.randn(5)

X = cp.Variable((n, n), symmetric=True)

constraints = [
    X >> 0,                      # PSD
    cp.trace(X) <= 10,           
]
for i in range(5):
    constraints.append(cp.trace(A[i] @ X) == b[i])

objective = cp.Minimize(cp.trace(C @ X))

problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.SCS)

print(f"Status: {problem.status}")
print(f"Optimal value: {problem.value}")
print(f"Trace of X: {np.trace(X.value):.4f}")
print(f"Optimal X:\n{np.round(X.value, 4)}")

Status: optimal
Optimal value: 1.5989829425855606
Trace of X: 1.8888
Optimal X:
[[ 0.1775 -0.1135  0.0502 -0.046   0.0829  0.0735  0.1913 -0.2177 -0.019
  -0.2146]
 [-0.1135  0.0934  0.001   0.0079 -0.07   -0.0689 -0.1479  0.1227  0.1251
   0.1569]
 [ 0.0502  0.001   0.0668 -0.0473 -0.0036 -0.014   0.0133 -0.0878  0.1743
  -0.0294]
 [-0.046   0.0079 -0.0473  0.0343 -0.0039  0.0036 -0.023   0.0735 -0.1121
   0.0353]
 [ 0.0829 -0.07   -0.0036 -0.0039  0.0526  0.0522  0.1103 -0.0882 -0.1011
  -0.1163]
 [ 0.0735 -0.0689 -0.014   0.0036  0.0522  0.0535  0.1062 -0.0728 -0.1267
  -0.1096]
 [ 0.1913 -0.1479  0.0133 -0.023   0.1103  0.1062  0.2376 -0.2142 -0.1594
  -0.2554]
 [-0.2177  0.1227 -0.0878  0.0735 -0.0882 -0.0728 -0.2142  0.2801 -0.0664
   0.2476]
 [-0.019   0.1251  0.1743 -0.1121 -0.1011 -0.1267 -0.1594 -0.0664  0.6149
   0.1296]
 [-0.2146  0.1569 -0.0294  0.0353 -0.1163 -0.1096 -0.2554  0.2476  0.1296
   0.2781]]


In [17]:
from pyscf import gto, scf

# Define molecule
mol = gto.Mole()
mol.atom = 'H 0 0 0; H 0 0 0.74'
mol.basis = '6-31g'
mol.build()

# Compute integrals
mf = scf.RHF(mol)
mf.kernel()

# Extract integrals
h1 = mf.get_hcore()
h2 = mf._eri  # Two-electron integrals
n_electrons = mol.nelectron
n_orbitals = mol.nao

converged SCF energy = -1.12675531719693


In [ ]:
"""
Variational 2-RDM (v2RDM) with DQG conditions for a small molecule,
in the spin-orbital basis. 

Conventions (r spin orbitals, real integrals):
  D1[p,q]     = <a_p^ a_q>
  D2[p,q,r,s] = <a_p^ a_q^ a_s a_r>
  H = sum_pq h[p,q] a_p^ a_q + 1/2 sum_pqrs V[p,q,r,s] a_p^ a_q^ a_s a_r
  E = sum_pq h[p,q] D1[p,q] + 1/2 sum_pqrs V[p,q,r,s] D2[p,q,r,s]
  Matrix layout idx(p,q) = p*r + q:
     D-matrix M[idx(p,q),idx(r,s)] = D2[p,q,r,s]
     Q[idx(p,q),idx(r,s)] = <a_p a_q a_s^ a_r^>            (2-hole)
     G[idx(p,q),idx(r,s)] = <a_q^ a_p a_r^ a_s>            (particle-hole)

Requires: numpy, pyscf, cvxpy  (and an SDP solver: CLARABEL)
"""
import numpy as np
from itertools import combinations
from pyscf import gto, scf, ao2mo, fci
import cvxpy as cp

# 1. Integrals: spatial MO -> spin-orbital basis
def spin_orbital_integrals(mol):
    """Return (h_so, V_so, n_so, n_elec, e_nuc) in physicist notation."""
    mf = scf.RHF(mol).run()
    mo = mf.mo_coeff
    nmo = mo.shape[1]

    h1_mo = mo.T @ mf.get_hcore() @ mo                       # (nmo,nmo)
    eri_chem = ao2mo.restore(1, ao2mo.kernel(mol, mo), nmo)  # (pq|rs) chemist
    eri_phys = eri_chem.transpose(0, 2, 1, 3)                # <pq|rs> = (pr|qs)

    r = 2 * nmo
    spin = lambda p: p % 2          # 0=alpha, 1=beta ; orbital = p // 2
    orb = lambda p: p // 2

    h_so = np.zeros((r, r))
    for p in range(r):
        for q in range(r):
            if spin(p) == spin(q):
                h_so[p, q] = h1_mo[orb(p), orb(q)]

    V_so = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s_ in range(r):
                for t in range(r):
                    # electron 1 carries (p,s_), electron 2 carries (q,t)
                    if spin(p) == spin(s_) and spin(q) == spin(t):
                        V_so[p, q, s_, t] = eri_phys[orb(p), orb(q), orb(s_), orb(t)]

    return h_so, V_so, r, mol.nelectron, mol.energy_nuc()


# 2. Fermionic operators + exact diagonalization 
def _below(det, p):
    return bin(det & ((1 << p) - 1)).count("1")

def _ann(p, det):
    return None if not (det >> p) & 1 else ((-1) ** _below(det, p), det & ~(1 << p))

def _cre(p, det):
    return None if (det >> p) & 1 else ((-1) ** _below(det, p), det | (1 << p))

def _apply(ops, state):
    for kind, orb in reversed(ops):
        new, f = {}, (_cre if kind == "c" else _ann)
        for det, amp in state.items():
            res = f(orb, det)
            if res:
                s, nd = res
                new[nd] = new.get(nd, 0.0) + s * amp
        state = new
    return state

def _expect(ops, psi):
    ket = _apply(ops, psi)
    return sum(psi.get(d, 0.0) * a for d, a in ket.items())

def exact_ground_state(h, V, r, N):
    basis = [sum(1 << i for i in occ) for occ in combinations(range(r), N)]
    pos = {d: i for i, d in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for j, det in enumerate(basis):
        ket, out = {det: 1.0}, {}
        for p in range(r):
            for q in range(r):
                if h[p, q]:
                    for d, a in _apply([("c", p), ("a", q)], ket).items():
                        out[d] = out.get(d, 0.0) + h[p, q] * a
        for p in range(r):
            for q in range(r):
                for s_ in range(r):
                    for t in range(r):
                        v = V[p, q, s_, t]
                        if v:
                            for d, a in _apply(
                                [("c", p), ("c", q), ("a", t), ("a", s_)], ket).items():
                                out[d] = out.get(d, 0.0) + 0.5 * v * a
        for d, a in out.items():
            H[pos[d], j] += a
    w, vecs = np.linalg.eigh(H)
    psi = {basis[i]: vecs[i, 0] for i in range(len(basis))}
    D1 = np.array([[_expect([("c", p), ("a", q)], psi) for q in range(r)]
                   for p in range(r)])
    D2 = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s_ in range(r):
                for t in range(r):
                    D2[p, q, s_, t] = _expect(
                        [("c", p), ("c", q), ("a", t), ("a", s_)], psi)
    return w[0], D1, D2


# 3. D/Q/G builders
def build_Q(D1, M, r, lib=np):
    I = np.eye(r)
    idx = lambda p, q: p * r + q
    rows = []
    for p in range(r):
        for q in range(r):
            row = []
            for u in range(r):
                for v in range(r):
                    const = (I[p, u] * I[q, v] - I[p, v] * I[q, u])
                    expr = (const
                            - I[q, v] * D1[p, u] + I[p, v] * D1[q, u]
                            + I[q, u] * D1[p, v] - I[p, u] * D1[q, v]
                            + M[idx(p, q), idx(u, v)])
                    row.append(expr)
            rows.append(row)
    return cp.bmat(rows) if lib is cp else np.array(rows, dtype=float)

def build_G(D1, M, r, lib=np):
    I = np.eye(r)
    idx = lambda p, q: p * r + q
    rows = []
    for p in range(r):
        for q in range(r):
            row = []
            for u in range(r):
                for v in range(r):
                    expr = I[p, u] * D1[q, v] - M[idx(q, u), idx(v, p)]
                    row.append(expr)
            rows.append(row)
    return cp.bmat(rows) if lib is cp else np.array(rows, dtype=float)


# 4. The SDP
def v2rdm_dqg(h_so, V_so, r, N, solver="CLARABEL"):
    r2 = r * r
    idx = lambda p, q: p * r + q
    M = cp.Variable((r2, r2), symmetric=True)   # D-matrix
    D1 = cp.Variable((r, r), symmetric=True)

    cons = [M >> 0, D1 >> 0,
            build_Q(D1, M, r, lib=cp) >> 0,
            build_G(D1, M, r, lib=cp) >> 0,
            cp.trace(D1) == N]

    # contraction  sum_x D2[p,x,q,x] = (N-1) D1[p,q]
    for p in range(r):
        for q in range(r):
            cons.append(cp.sum([M[idx(p, x), idx(q, x)] for x in range(r)])
                        == (N - 1) * D1[p, q])

    # antisymmetry of D2
    for p in range(r):
        for q in range(r):
            for u in range(r):
                for v in range(r):
                    cons.append(M[idx(p, q), idx(u, v)] == -M[idx(q, p), idx(u, v)])
                    cons.append(M[idx(p, q), idx(u, v)] == -M[idx(p, q), idx(v, u)])

    h_mat = h_so
    V_mat = V_so.reshape(r2, r2)
    E = cp.sum(cp.multiply(h_mat, D1)) + 0.5 * cp.sum(cp.multiply(V_mat, M))
    prob = cp.Problem(cp.Minimize(E), cons)
    prob.solve(solver=solver, verbose=False)
    return prob.value, prob.status, D1.value, M.value


# 5. Run on H2 / STO-3G
if __name__ == "__main__":
    mol = gto.M(atom="H 0 0 0; H 0 0 1.00", basis="sto-3g", verbose=0)
    h_so, V_so, r, N, e_nuc = spin_orbital_integrals(mol)
    print(f"spin orbitals r={r}, electrons N={N}, E_nuc={e_nuc:.10f}")

    # --- ground truth: exact diag, cross-checked vs pyscf FCI ---
    E_elec_exact, D1x, D2x = exact_ground_state(h_so, V_so, r, N)
    mf = scf.RHF(mol).run()
    nmo = mf.mo_coeff.shape[1]
    h1_mo = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
    eri_mo = ao2mo.restore(1, ao2mo.kernel(mol, mf.mo_coeff), nmo)
    e_fci, _ = fci.direct_spin1.kernel(h1_mo, eri_mo, nmo, N)
    assert abs(E_elec_exact - e_fci) < 1e-8, (E_elec_exact, e_fci)
    print(f"exact-diag electronic energy matches pyscf FCI  "
          f"(diff {abs(E_elec_exact - e_fci):.1e})")

    # --- assert the exact RDMs satisfy every constraint we impose ---
    Mx = D2x.reshape(r * r, r * r)
    E_form = np.einsum("pq,pq->", h_so, D1x) + 0.5 * np.einsum("pqrs,pqrs->", V_so, D2x)
    assert abs(E_form - E_elec_exact) < 1e-9
    assert abs(np.trace(D1x) - N) < 1e-9
    assert abs(np.trace(Mx) - N * (N - 1)) < 1e-9
    assert np.abs(np.einsum("prqr->pq", D2x) - (N - 1) * D1x).max() < 1e-9
    assert np.linalg.eigvalsh(Mx).min() > -1e-9
    assert np.linalg.eigvalsh(build_Q(D1x, Mx, r)).min() > -1e-9
    assert np.linalg.eigvalsh(build_G(D1x, Mx, r)).min() > -1e-9
    print("exact RDMs satisfy energy, trace, contraction, and D/Q/G PSD  [OK]")

    # --- the SDP: DQG lower bound ---
    E_sdp_elec, status, _, _ = v2rdm_dqg(h_so, V_so, r, N)
    E_sdp = E_sdp_elec + e_nuc
    E_fci_total = e_fci + e_nuc
    print(f"\nSDP status        : {status}")
    print(f"v2RDM-DQG energy  : {E_sdp:.10f} Ha")
    print(f"FCI energy        : {E_fci_total:.10f} Ha")
    print(f"gap (FCI - v2RDM) : {E_fci_total - E_sdp:.2e} Ha  "
          f"(>=0; ~0 means DQG is tight for this 2e system)")

spin orbitals r=4, electrons N=2, E_nuc=0.5291772109
exact-diag electronic energy matches pyscf FCI  (diff 4.4e-16)
exact RDMs satisfy energy, trace, contraction, and D/Q/G PSD  [OK]

SDP status        : optimal_inaccurate
v2RDM-DQG energy  : -1.1011503064 Ha
FCI energy        : -1.1011503302 Ha
gap (FCI - v2RDM) : -2.38e-08 Ha  (>=0; ~0 means DQG is tight for this 2e system)


/tmp/ipykernel_2478/1342755892.py:197: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  prob.solve(solver=solver, verbose=False)


In [ ]:
"""
Boundary-point method (Malick-Povh-Rendl-Wiegele):
ADMM on the DUAL of  min <C,X> s.t. A(X)=b, X >= 0.

Dual:  max b^T y  s.t.  Z = C - A*(y) >= 0.
The primal X appears as the Lagrange multiplier of the dual equality, so one
algorithm produces primal and dual simultaneously. Per iteration:
    rhs = (1/sigma)(b - A(X)) - A(Z) + A(C)
    solve (A A*) y = rhs                      # one linear solve, prefactored
    W   = C - A*(y) - X/sigma
    eigh(W):  Z = W_+ ,  X = -sigma * W_-     # one eigendecomposition
"""
import numpy as np

def bpsdp(C, As, b, sigma=1.0, iters=4000, tol=1e-9):
    n, m = C.shape[0], len(As)
    Avecs = np.array([A.reshape(-1) for A in As])          # m x n^2
    G = Avecs @ Avecs.T                                    # Gram A A*
    # regularize for possibly-redundant constraints 
    Gfac = np.linalg.cholesky(G + 1e-10 * np.eye(m))
    cvec = C.reshape(-1)
    Ac = Avecs @ cvec                                      # A(C)
    Astar = lambda y: (Avecs.T @ y).reshape(n, n)
    Aop = lambda X: Avecs @ X.reshape(-1)

    X = np.eye(n)
    Z = np.zeros((n, n))
    for k in range(iters):
        rhs = (b - Aop(X)) / sigma - Aop(Z) + Ac
        y = np.linalg.solve(Gfac.T, np.linalg.solve(Gfac, rhs))
        W = C - Astar(y) - X / sigma
        W = 0.5 * (W + W.T)
        w, U = np.linalg.eigh(W)
        wp = np.clip(w, 0, None)
        wn = np.clip(w, None, 0)
        Z = (U * wp) @ U.T
        X = -sigma * (U * wn) @ U.T
        if k % 100 == 0 or k == iters - 1:
            pinf = np.linalg.norm(Aop(X) - b)
            gap = abs(np.sum(C * X) - b @ y)
            if pinf < tol and gap < tol:
                break
    return {"primal": float(np.sum(C * X)), "dual": float(b @ y),
            "X": X, "y": y, "Z": Z,
            "pinf": float(np.linalg.norm(Aop(X) - b)),
            "gap": float(abs(np.sum(C * X) - b @ y)),
            "compl": float(np.sum(X * Z)),
            "min_eig_X": float(np.linalg.eigvalsh(X).min()),
            "min_eig_Z": float(np.linalg.eigvalsh(Z).min()),
            "iters": k + 1}


# ---- bounded instance: primal-feasible X* and strictly dual-feasible (y0,Z0) ----
rng = np.random.default_rng(1)
n, m = 8, 6
As = []
for _ in range(m):
    Aj = rng.standard_normal((n, n)); As.append(0.5 * (Aj + Aj.T))
# primal feasible point -> defines b
L = rng.standard_normal((n, n)); Xstar = L @ L.T + np.eye(n)
b = np.array([np.sum(A * Xstar) for A in As])
# strictly dual-feasible point -> defines C = A*(y0) + Z0, Z0 >> 0
y0 = rng.standard_normal(m)
Mz = rng.standard_normal((n, n)); Z0 = Mz @ Mz.T + np.eye(n)
Cm = sum(y0[j] * As[j] for j in range(m)) + Z0

res = bpsdp(Cm, As, b)
print(f"iters        : {res['iters']}")
print(f"primal <C,X> : {res['primal']:+.10f}")
print(f"dual   b^T y : {res['dual']:+.10f}")
print(f"duality gap  : {res['gap']:.2e}")
print(f"primal infeas: {res['pinf']:.2e}")
print(f"compl <X,Z>  : {res['compl']:.2e}")
print(f"min eig X    : {res['min_eig_X']:+.2e}  (>=0)")
print(f"min eig Z    : {res['min_eig_Z']:+.2e}  (>=0)")

iters        : 4000
primal <C,X> : +191.3742906990
dual   b^T y : +191.3742907031
duality gap  : 4.02e-09
primal infeas: 6.34e-10
compl <X,Z>  : 1.42e-14
min eig X    : -6.37e-15  (>=0)
min eig Z    : -2.45e-15  (>=0)


In [29]:
"""
Cumulant (connected) decomposition of the 2-RDM, and how the SAME cumulant
appears in the Q (two-hole) and G (particle-hole) metrics.

Conventions (spin orbitals, real):
  D1[p,q]     = <a_p^ a_q>                      particle 1-RDM
  Dh = I - D1                                   hole 1-RDM,  Dh[p,q] = <a_p a_q^>
  D2[p,q,r,s] = <a_p^ a_q^ a_s a_r>             2-RDM (trace = N(N-1))

Wedge (Grassmann) product of one-particle matrices, in the normalization where
trace(2-RDM) = N(N-1)  (coefficient 1; other refs carry 1/2 or 1/4):
  (A ^ B)[p,q,r,s] = 1/2 ( A[p,r]B[q,s] - A[p,s]B[q,r]
                          + B[p,r]A[q,s] - B[p,s]A[q,r] )
  for A=B this is just  A[p,r]A[q,s] - A[p,s]A[q,r].

Central identities (verified below):
  2-RDM :  D2 =  D1 ^ D1            + L
  2-hole:  Q  =  Dh ^ Dh            + L                 (SAME cumulant L)
  p-hole:  G  =  Dh[p,r]D1[q,s] + D1[p,q]D1[r,s]  -  L[q,r,s,p]
  cumulant trace:  sum_r L[p,r,q,r] = -(D1 (I - D1))[p,q]
The mean-field (1-RDM-only) background differs between P/Q/G; the correlation
content L is one and the same object.
"""
import numpy as np
from itertools import combinations

# ---------- fermionic engine (same as before) ----------
def _below(d, p): return bin(d & ((1 << p) - 1)).count("1")
def _ann(p, d):  return None if not (d >> p) & 1 else ((-1) ** _below(d, p), d & ~(1 << p))
def _cre(p, d):  return None if (d >> p) & 1 else ((-1) ** _below(d, p), d | (1 << p))
def _apply(ops, st):
    for k, o in reversed(ops):
        new, f = {}, (_cre if k == "c" else _ann)
        for d, a in st.items():
            r = f(o, d)
            if r: s, nd = r; new[nd] = new.get(nd, 0.0) + s * a
        st = new
    return st
def _exp(ops, psi):
    ket = _apply(ops, psi)
    return sum(psi.get(d, 0.0) * a for d, a in ket.items())

def rdms(psi, r):
    D1 = np.array([[_exp([("c", p), ("a", q)], psi) for q in range(r)] for p in range(r)])
    D2 = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s in range(r):
                for t in range(r):
                    D2[p, q, s, t] = _exp([("c", p), ("c", q), ("a", t), ("a", s)], psi)
    return D1, D2

def ground_state(h, V, r, N):
    basis = [sum(1 << i for i in occ) for occ in combinations(range(r), N)]
    pos = {d: i for i, d in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for j, det in enumerate(basis):
        ket, out = {det: 1.0}, {}
        for p in range(r):
            for q in range(r):
                if h[p, q]:
                    for d, a in _apply([("c", p), ("a", q)], ket).items():
                        out[d] = out.get(d, 0.0) + h[p, q] * a
        for p in range(r):
            for q in range(r):
                for s in range(r):
                    for t in range(r):
                        if V[p, q, s, t]:
                            for d, a in _apply([("c", p), ("c", q), ("a", t), ("a", s)], ket).items():
                                out[d] = out.get(d, 0.0) + 0.5 * V[p, q, s, t] * a
        for d, a in out.items():
            H[pos[d], j] += a
    w, vec = np.linalg.eigh(H)
    return {basis[i]: vec[i, 0] for i in range(len(basis))}

# ---------- wedge, and the verified Q/G builders ----------
def wedge(A, B):
    """Grassmann product of one-particle matrices (coefficient-1 normalization)."""
    t = (np.einsum("pr,qs->pqrs", A, B) - np.einsum("ps,qr->pqrs", A, B)
         + np.einsum("pr,qs->pqrs", B, A) - np.einsum("ps,qr->pqrs", B, A))
    return 0.5 * t

def build_Q(D1, D2, r):
    I = np.eye(r); Q = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for u in range(r):
                for v in range(r):
                    Q[p, q, u, v] = (I[p, u] * I[q, v] - I[p, v] * I[q, u]
                                     - I[q, v] * D1[p, u] + I[p, v] * D1[q, u]
                                     + I[q, u] * D1[p, v] - I[p, u] * D1[q, v]
                                     + D2[p, q, u, v])
    return Q

def build_G(D1, D2, r):
    I = np.eye(r); G = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for u in range(r):
                for v in range(r):
                    G[p, q, u, v] = I[p, u] * D1[q, v] - D2[q, u, v, p]
    return G


def report(tag, psi, r, N):
    D1, D2 = rdms(psi, r)
    I = np.eye(r); Dh = I - D1

    # --- cumulant: connected part of the 2-RDM ---
    L = D2 - wedge(D1, D1)

    print(f"\n===== {tag} =====")
    print(f"natural occupations (eig D1): {np.round(np.linalg.eigvalsh(D1), 4)}")
    print(f"||cumulant L||_F           : {np.linalg.norm(L):.4e}   "
          f"(0  <=>  single determinant)")

    # D2 = D1^D1 + L  (definitional, but confirm antisymmetry of L)
    asymL = max(np.abs(L + L.transpose(1, 0, 2, 3)).max(),
                np.abs(L + L.transpose(0, 1, 3, 2)).max())
    print(f"L antisymmetric             : max|L+swap| = {asymL:.2e}")

    # Q = Dh^Dh + L   (same cumulant!)
    Q = build_Q(D1, D2, r)
    Q_dec = wedge(Dh, Dh) + L
    print(f"Q == Dh^Dh + L              : max|diff| = {np.abs(Q - Q_dec).max():.2e}")

    # G = Dh[p,r]D1[q,s] + D1[p,q]D1[r,s] - L[q,r,s,p]
    G = build_G(D1, D2, r)
    Gmf = (np.einsum("pr,qs->pqrs", Dh, D1) + np.einsum("pq,rs->pqrs", D1, D1))
    G_dec = Gmf - L.transpose(1, 2, 3, 0)        # L[q,r,s,p] for entry [p,q,r,s]
    print(f"G == G_mf(D1) - L[q,r,s,p]  : max|diff| = {np.abs(G - G_dec).max():.2e}")

    # cumulant contraction <-> non-idempotency of D1
    contrL = np.einsum("prqr->pq", L)
    target = -(D1 @ (I - D1))
    print(f"sum_r L[p,r,q,r] == -D1(I-D1): max|diff| = {np.abs(contrL - target).max():.2e}")
    print(f"  trace of that = -tr(D1(I-D1)) = {-np.trace(D1 @ (I - D1)):+.4f}  "
          f"(0 for idempotent/HF; >0 measures correlation)")


if __name__ == "__main__":
    r, N = 4, 2

    # (A) a single Slater determinant: occupy spin-orbitals 0 and 1 -> cumulant must vanish
    det_state = {0b0011: 1.0}
    report("single Slater determinant (uncorrelated)", det_state, r, N)

    # (B) a correlated ground state of a random 2-electron Hamiltonian
    rng = np.random.default_rng(7)
    h = rng.standard_normal((r, r)); h = 0.5 * (h + h.T)
    W = rng.standard_normal((r, r, r, r))
    V = 0.5 * (W + W.transpose(2, 3, 0, 1))
    V = 0.5 * (V + V.transpose(1, 0, 3, 2))
    report("correlated random 2-electron state", ground_state(h, V, r, N), r, N)


===== single Slater determinant (uncorrelated) =====
natural occupations (eig D1): [0. 0. 1. 1.]
||cumulant L||_F           : 0.0000e+00   (0  <=>  single determinant)
L antisymmetric             : max|L+swap| = 0.00e+00
Q == Dh^Dh + L              : max|diff| = 0.00e+00
G == G_mf(D1) - L[q,r,s,p]  : max|diff| = 0.00e+00
sum_r L[p,r,q,r] == -D1(I-D1): max|diff| = 0.00e+00
  trace of that = -tr(D1(I-D1)) = -0.0000  (0 for idempotent/HF; >0 measures correlation)

===== correlated random 2-electron state =====
natural occupations (eig D1): [0.0201 0.0201 0.9799 0.9799]
||cumulant L||_F           : 4.0880e-01   (0  <=>  single determinant)
L antisymmetric             : max|L+swap| = 2.78e-17
Q == Dh^Dh + L              : max|diff| = 5.55e-17
G == G_mf(D1) - L[q,r,s,p]  : max|diff| = 1.11e-16
sum_r L[p,r,q,r] == -D1(I-D1): max|diff| = 3.71e-16
  trace of that = -tr(D1(I-D1)) = -0.0789  (0 for idempotent/HF; >0 measures correlation)


In [ ]:
"""
Demonstrates the one subtle point in reconstructing  D2 = D1^D1 + Lambda
from measured moments: the wedge part D1^D1 is quadratic in the 1-RDM, so
plugging a single noisy estimate D1_hat into the wedge gives a biased estimate
of the mean-field part (and hence of the cumulant). Splitting the shots into
two independent halves and cross-multiplying removes the bias.

Pure numpy; runnable. (The SDP purification step is shown as structure in the
LaTeX note and reuses the verified build_Q / build_G from earlier.)
"""
import numpy as np
from itertools import combinations

# ---- minimal code to get a genuine correlated 2e state ----
def _below(d, p): return bin(d & ((1 << p) - 1)).count("1")
def _ann(p, d):  return None if not (d >> p) & 1 else ((-1) ** _below(d, p), d & ~(1 << p))
def _cre(p, d):  return None if (d >> p) & 1 else ((-1) ** _below(d, p), d | (1 << p))
def _apply(ops, st):
    for k, o in reversed(ops):
        new, f = {}, (_cre if k == "c" else _ann)
        for d, a in st.items():
            r = f(o, d)
            if r: s, nd = r; new[nd] = new.get(nd, 0.0) + s * a
        st = new
    return st
def _exp(ops, psi): 
    ket = _apply(ops, psi); return sum(psi.get(d, 0.0) * a for d, a in ket.items())

def exact_rdms(r=4, N=2, seed=7):
    rng = np.random.default_rng(seed)
    h = rng.standard_normal((r, r)); h = 0.5 * (h + h.T)
    W = rng.standard_normal((r, r, r, r))
    V = 0.5 * (W + W.transpose(2, 3, 0, 1)); V = 0.5 * (V + V.transpose(1, 0, 3, 2))
    basis = [sum(1 << i for i in occ) for occ in combinations(range(r), N)]
    pos = {d: i for i, d in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for j, det in enumerate(basis):
        ket, out = {det: 1.0}, {}
        for p in range(r):
            for q in range(r):
                if h[p, q]:
                    for d, a in _apply([("c", p), ("a", q)], ket).items():
                        out[d] = out.get(d, 0.0) + h[p, q] * a
        for p in range(r):
            for q in range(r):
                for s in range(r):
                    for t in range(r):
                        if V[p, q, s, t]:
                            for d, a in _apply([("c", p), ("c", q), ("a", t), ("a", s)], ket).items():
                                out[d] = out.get(d, 0.0) + 0.5 * V[p, q, s, t] * a
        for d, a in out.items(): H[pos[d], j] += a
    w, vec = np.linalg.eigh(H); psi = {basis[i]: vec[i, 0] for i in range(len(basis))}
    D1 = np.array([[_exp([("c", p), ("a", q)], psi) for q in range(r)] for p in range(r)])
    D2 = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s in range(r):
                for t in range(r):
                    D2[p, q, s, t] = _exp([("c", p), ("c", q), ("a", t), ("a", s)], psi)
    return D1, D2

def wedge(A, B):
    return 0.5 * (np.einsum("pr,qs->pqrs", A, B) - np.einsum("ps,qr->pqrs", A, B)
                  + np.einsum("pr,qs->pqrs", B, A) - np.einsum("ps,qr->pqrs", B, A))

def noisy(M, sigma, rng):
    """symmetric additive shot-noise model on a 1-RDM estimate."""
    E = rng.standard_normal(M.shape) * sigma
    E = 0.5 * (E + E.T)
    return M + E

# ---------------------------------------------------------------
# Monte-Carlo: bias of the cumulant's mean-field part
#   naive : wedge(D1_hat, D1_hat)            with one noisy estimate
#   split : wedge(D1_hatA, D1_hatB)          two independent half-shot estimates
# ---------------------------------------------------------------
D1, D2 = exact_rdms()
true_wedge = wedge(D1, D1)
true_cumulant = D2 - true_wedge

rng = np.random.default_rng(0)
sigma = 0.05                # per-element 1-RDM noise from finite shots
trials = 40000

naive_acc = np.zeros_like(true_wedge)
split_acc = np.zeros_like(true_wedge)
for _ in range(trials):
    # one estimate from all shots vs two estimates from disjoint halves
    # (halves have sqrt(2) larger noise each, modelled by sigma*sqrt(2))
    D1_full = noisy(D1, sigma, rng)
    D1_A = noisy(D1, sigma * np.sqrt(2), rng)
    D1_B = noisy(D1, sigma * np.sqrt(2), rng)
    naive_acc += wedge(D1_full, D1_full)
    split_acc += wedge(D1_A, D1_B)

naive_mean = naive_acc / trials
split_mean = split_acc / trials

print("Cumulant mean-field part  D1^D1  (estimator bias under shot noise)")
print(f"  sigma (1-RDM)            : {sigma}")
print(f"  ||bias|| naive wedge     : {np.linalg.norm(naive_mean - true_wedge):.4e}")
print(f"  ||bias|| sample-split    : {np.linalg.norm(split_mean - true_wedge):.4e}")
print(f"  -> naive over-estimates the diagonal by ~sigma^2 per element; "
      f"split is unbiased.")

# effect propagates directly into the cumulant estimate Lambda = D2 - wedge
print("\nResulting cumulant error (mean-field bias leaks into Lambda):")
print(f"  ||Lambda_naive - Lambda||: {np.linalg.norm((D2 - naive_mean) - true_cumulant):.4e}")
print(f"  ||Lambda_split - Lambda||: {np.linalg.norm((D2 - split_mean) - true_cumulant):.4e}")

Cumulant mean-field part  D1^D1  (estimator bias under shot noise)
  sigma (1-RDM)            : 0.05
  ||bias|| naive wedge     : 6.9730e-03
  ||bias|| sample-split    : 1.5254e-03
  -> naive over-estimates the diagonal by ~sigma^2 per element; split is unbiased.

Resulting cumulant error (mean-field bias leaks into Lambda):
  ||Lambda_naive - Lambda||: 6.9730e-03
  ||Lambda_split - Lambda||: 1.5254e-03


In [ ]:
"""
Eigenvalues of the reconstructed 2-cumulant as orbital-basis-invariant
correlation features for ML.

Matricize the cumulant  L[p,q,r,s]  ->  L_mat[idx(p,q), idx(r,s)],  idx(p,q)=p*r+q.
L_mat is real-symmetric (Hermiticity of D2 + symmetry of the wedge), so its
eigenvalues are real. Under any single-particle unitary R (orbital rotation),
   D1 -> R D1 R^T ,   D2 -> (R⊗R) D2 (R⊗R)^T ,   hence  L -> (R⊗R) L (R⊗R)^T ,
so L_mat is conjugated by a unitary and its spectrum is invariant. That makes the
spectrum (and its moments tr L_mat^k) a gauge-free descriptor of correlation.

Two exact constraints worth knowing for feature design:
   sum_k lambda_k = tr L_mat = -tr[D1(I-D1)]   <= 0     (fixed by the 1-RDM)
   lambda_k = 0 outside the antisymmetric 2-particle subspace (rank <= r(r-1)/2)
"""
import numpy as np
from itertools import combinations

# ---------- fermionic part ----------
def _below(d, p): return bin(d & ((1 << p) - 1)).count("1")
def _ann(p, d):  return None if not (d >> p) & 1 else ((-1) ** _below(d, p), d & ~(1 << p))
def _cre(p, d):  return None if (d >> p) & 1 else ((-1) ** _below(d, p), d | (1 << p))
def _apply(ops, st):
    for k, o in reversed(ops):
        new, f = {}, (_cre if k == "c" else _ann)
        for d, a in st.items():
            r = f(o, d)
            if r: s, nd = r; new[nd] = new.get(nd, 0.0) + s * a
        st = new
    return st
def _exp(ops, psi):
    ket = _apply(ops, psi); return sum(psi.get(d, 0.0) * a for d, a in ket.items())

def ground_rdms(h, V, r, N):
    basis = [sum(1 << i for i in occ) for occ in combinations(range(r), N)]
    pos = {d: i for i, d in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for j, det in enumerate(basis):
        ket, out = {det: 1.0}, {}
        for p in range(r):
            for q in range(r):
                if h[p, q]:
                    for d, a in _apply([("c", p), ("a", q)], ket).items():
                        out[d] = out.get(d, 0.0) + h[p, q] * a
        for p in range(r):
            for q in range(r):
                for s in range(r):
                    for t in range(r):
                        if V[p, q, s, t]:
                            for d, a in _apply([("c", p), ("c", q), ("a", t), ("a", s)], ket).items():
                                out[d] = out.get(d, 0.0) + 0.5 * V[p, q, s, t] * a
        for d, a in out.items(): H[pos[d], j] += a
    w, vec = np.linalg.eigh(H)
    psi = {basis[i]: vec[i, 0] for i in range(len(basis))}
    D1 = np.array([[_exp([("c", p), ("a", q)], psi) for q in range(r)] for p in range(r)])
    D2 = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s in range(r):
                for t in range(r):
                    D2[p, q, s, t] = _exp([("c", p), ("c", q), ("a", t), ("a", s)], psi)
    return w[0], D1, D2

# ---------- cumulant + its spectral features ----------
def wedge(A, B):
    return 0.5 * (np.einsum("pr,qs->pqrs", A, B) - np.einsum("ps,qr->pqrs", A, B)
                  + np.einsum("pr,qs->pqrs", B, A) - np.einsum("ps,qr->pqrs", B, A))

def cumulant(D1, D2):
    return D2 - wedge(D1, D1)

def cumulant_spectrum(D1, D2):
    r = D1.shape[0]
    L = cumulant(D1, D2)
    Lmat = L.reshape(r * r, r * r)
    Lmat = 0.5 * (Lmat + Lmat.T)             
    return np.linalg.eigvalsh(Lmat)

def spectral_moments(eigs, kmax=4):
    """tr(L_mat^k) = sum lambda^k : fixed-length, basis- and size-agnostic features."""
    return np.array([np.sum(eigs ** k) for k in range(1, kmax + 1)])

# ---------- orbital rotation (acts on RDM tensors) ----------
def rotate_rdms(D1, D2, R):
    D1r = R @ D1 @ R.T
    D2r = np.einsum("pa,qb,abcd,rc,sd->pqrs", R, R, D2, R, R)
    return D1r, D2r

# ---------- model: 2-site Hubbard dimer, half filling (4 spin-orb, 2 e) ----------
# spin orbital index = 2*site + spin ; site0 = orbs(0,1), site1 = orbs(2,3)
def hubbard_dimer(t, U):
    r = 4
    h = np.zeros((r, r))
    for spin in (0, 1):
        a, b = 0 * 2 + spin, 1 * 2 + spin
        h[a, b] = h[b, a] = -t
    V = np.zeros((r, r, r, r))
    for site in (0, 1):
        u, d = site * 2 + 0, site * 2 + 1            # up,down orbitals on this site
        # U n_up n_down  -> these entries reproduce U <a_u^ a_d^ a_d a_u>
        V[u, d, u, d] = V[d, u, d, u] = U / 2
        V[d, u, u, d] = V[u, d, d, u] = -U / 2
    return h, V, r, 2


if __name__ == "__main__":
    np.set_printoptions(precision=4, suppress=True)
    t = 1.0

    # sanity: U=0 dimer ground state energy is -2t
    E0, D1, D2 = ground_rdms(*hubbard_dimer(t, 0.0))
    print(f"U=0 dimer energy = {E0:+.4f}  (expect {-2*t:+.4f})\n")

    print("Hubbard dimer (t=1): cumulant spectrum vs correlation strength U")
    print(f"{'U':>5} | {'occupations':>22} | {'sum lambda':>10} | "
          f"{'-tr D1(I-D1)':>12} | nonzero cumulant eigenvalues")
    print("-" * 100)
    feature_rows = []
    for U in [0.0, 1.0, 2.0, 4.0, 8.0, 16.0]:
        E0, D1, D2 = ground_rdms(*hubbard_dimer(t, U))
        occ = np.linalg.eigvalsh(D1)
        eigs = cumulant_spectrum(D1, D2)
        nz = eigs[np.abs(eigs) > 1e-9]
        s = np.sum(eigs)
        tr_id = -np.trace(D1 @ (np.eye(4) - D1))
        moms = spectral_moments(eigs)
        feature_rows.append((U, moms))
        print(f"{U:5.1f} | {np.array2string(occ):>22} | {s:10.4f} | "
              f"{tr_id:12.4f} | {np.array2string(np.sort(nz))}")

    # ---- orbital-rotation invariance of the spectrum (the ML-relevant property) ----
    print("\nOrbital-rotation invariance check (U=4):")
    E0, D1, D2 = ground_rdms(*hubbard_dimer(t, 4.0))
    eigs0 = cumulant_spectrum(D1, D2)
    rng = np.random.default_rng(3)
    max_dev = 0.0
    for _ in range(5):
        A = rng.standard_normal((4, 4)); Q, _ = np.linalg.qr(A)   # random O(4)
        D1r, D2r = rotate_rdms(D1, D2, Q)
        eigsr = cumulant_spectrum(D1r, D2r)
        max_dev = max(max_dev, np.abs(np.sort(eigsr) - np.sort(eigs0)).max())
    print(f"  max |spectrum(rotated) - spectrum(original)| over 5 rotations = {max_dev:.2e}")
    print("  -> the spectrum is gauge-free: independent of the orbital basis.")

    # ---- fixed-length feature vector (spectral moments) regardless of system size ----
    print("\nFixed-length invariant feature vector  [tr L, tr L^2, tr L^3, tr L^4]:")
    for U, m in feature_rows:
        print(f"  U={U:4.1f}: {np.array2string(m)}")

U=0 dimer energy = -2.0000  (expect -2.0000)

Hubbard dimer (t=1): cumulant spectrum vs correlation strength U
    U |            occupations | sum lambda | -tr D1(I-D1) | nonzero cumulant eigenvalues
----------------------------------------------------------------------------------------------------
  0.0 |          [0. 0. 1. 1.] |     0.0000 |      -0.0000 | []
  1.0 | [0.0149 0.0149 0.9851 0.9851] |    -0.0588 |      -0.0588 | [-0.2131 -0.0294 -0.0294 -0.0294 -0.0294  0.2719]
  2.0 | [0.0528 0.0528 0.9472 0.9472] |    -0.2000 |      -0.2000 | [-0.3472 -0.1    -0.1    -0.1    -0.1     0.5472]
  4.0 | [0.1464 0.1464 0.8536 0.8536] |    -0.5000 |      -0.5000 | [-0.4571 -0.25   -0.25   -0.25   -0.25    0.9571]
  8.0 | [0.2764 0.2764 0.7236 0.7236] |    -0.8000 |      -0.8000 | [-0.4944 -0.4    -0.4    -0.4    -0.4     1.2944]
 16.0 | [0.3787 0.3787 0.6213 0.6213] |    -0.9412 |      -0.9412 | [-0.4996 -0.4706 -0.4706 -0.4706 -0.4706  1.4407]

Orbital-rotation invariance check (U=4):
  

In [ ]:
"""
ml_pipeline.py
--------------
Three models on the same dataset, same budget:

  Model A  : f(U/t) -> E0          energy regression baseline
  Model B  : g(U/t) -> vec(Lambda) raw cumulant regression  (Idea I)
  Model C  : h(U/t) -> moments     spectral moment regression (Idea II)

All three use the same MLP backbone via sklearn.
Energy from B and C is derived by contracting the predicted 2-RDM with h, V.
The key experiments:

  Exp 1  : in-distribution accuracy (train and test on same U/t range)
  Exp 2  : transferability (train on weak correlation, test on strong)
  Exp 3  : Idea II cross-system (train on 4-site, test on 6-site — same model)
"""

import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.kernel_ridge import KernelRidge
import warnings
warnings.filterwarnings('ignore')

from rdm_core import (
    wedge, cumulant, spectral_moments, generate_hubbard_dataset,
    hubbard_chain, exact_diag, noisy_rdm, sample_split_wedge
)


# ── feature extraction ────────────────────────────────────────────────────────

def features_scalar(entry):
    """Input feature: just U/t (scalar). Simplest possible input."""
    return np.array([entry['U_over_t']])

def target_energy(entry):
    return np.array([entry['E0_per_site']])

def target_cumulant_flat(entry):
    """Idea I: upper triangle of cumulant (antisymmetry reduces storage)."""
    cum = entry['cumulant']
    n = cum.shape[0]
    # Keep only independent entries: p<=q, r<=s, (p,q)<=(r,s)
    # For simplicity use the full flattened upper-triangular of the matrix form
    Lmat = cum.reshape(n*n, n*n)
    Lmat = 0.5*(Lmat + Lmat.T)
    idx = np.triu_indices(n*n)
    return Lmat[idx]

def target_spectral_moments(entry, k_max=6):
    """Idea II: spectral moments [tr L, tr L^2, ..., tr L^k_max]."""
    return entry['spectral_moments'][:k_max]


# ── energy from predicted 2-RDM ──────────────────────────────────────────────

def energy_from_prediction(pred_vec, D1_exact, h, V, n_orb, mode='cumulant'):
    """
    Reconstruct energy from a model's prediction.

    mode='cumulant': pred_vec is flattened upper-tri of Lmat
                     -> reconstruct D2 = D1^D1 + Lambda, evaluate energy
    mode='moments' : pred_vec is spectral moments (cannot reconstruct full D2,
                     so this returns None — moments don't uniquely fix the energy)
    """
    if mode == 'moments':
        return None   # spectral moments don't uniquely determine energy

    # Reconstruct Lmat from upper-triangular vector
    n = n_orb
    Lmat = np.zeros((n*n, n*n))
    idx = np.triu_indices(n*n)
    Lmat[idx] = pred_vec
    Lmat = Lmat + Lmat.T - np.diag(np.diag(Lmat))  # symmetrize
    L = Lmat.reshape(n, n, n, n)

    # D2 = D1 ^ D1 + Lambda (use exact D1 — it's cheaply measured)
    D2_pred = wedge(D1_exact, D1_exact) + L

    # Energy
    E = np.einsum('pq,pq->', h, D1_exact)
    E += 0.5 * np.einsum('pqrs,pqrs->', V, D2_pred)
    return E


# ── MLP builder ──────────────────────────────────────────────────────────────

def make_mlp(hidden=(64, 64), max_iter=2000, random_state=42):
    """Standard sklearn MLP with input standardisation."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_layer_sizes=hidden,
            activation='tanh',
            max_iter=max_iter,
            random_state=random_state,
            learning_rate_init=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=50,
        ))
    ])

def make_krr():
    """Kernel ridge regression — good baseline for small datasets."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('krr', KernelRidge(kernel='rbf', alpha=1e-3, gamma=1.0))
    ])


# ── experiment runner ─────────────────────────────────────────────────────────

class CumulantMLExperiment:
    def __init__(self, n_sites, t=1.0):
        self.n_sites = n_sites
        self.t = t
        self.h, self.V, self.n_orb, self.n_elec = hubbard_chain(n_sites, t, U=0)
        # Recompute h,V at each U, but store structure
        self._h_base = None

    def _get_hV(self, U):
        h, V, _, _ = hubbard_chain(self.n_sites, self.t, U)
        return h, V

    def run_transferability_experiment(
            self,
            U_train, U_test,
            n_shots_list=None,
            verbose=True):
        """
        Core experiment: train on U_train, evaluate on U_test.
        Compares Model A (energy) vs Model B (cumulant) vs Model C (moments).

        n_shots_list: if provided, also evaluates Model B under shadow noise
                      at each shot budget (simulates quantum device).
        """
        print(f"\n{'='*60}")
        print(f"Transferability experiment: {self.n_sites}-site Hubbard")
        print(f"  Train: U/t in {[f'{u:.1f}' for u in U_train[:3]]}...{[f'{u:.1f}' for u in U_train[-2:]]}")
        print(f"  Test:  U/t in {[f'{u:.1f}' for u in U_test[:3]]}...{[f'{u:.1f}' for u in U_test[-2:]]}")
        print(f"{'='*60}")

        # Generate datasets
        print("\nGenerating training data...")
        train_data = generate_hubbard_dataset(self.n_sites, U_train, self.t, verbose=verbose)
        print("Generating test data...")
        test_data  = generate_hubbard_dataset(self.n_sites, U_test,  self.t, verbose=verbose)

        # Build feature/target arrays
        X_tr = np.array([features_scalar(d) for d in train_data])
        X_te = np.array([features_scalar(d) for d in test_data])

        yA_tr = np.array([target_energy(d) for d in train_data]).ravel()
        yA_te = np.array([target_energy(d) for d in test_data]).ravel()

        yB_tr = np.array([target_cumulant_flat(d) for d in train_data])
        yB_te = np.array([target_cumulant_flat(d) for d in test_data])

        yC_tr = np.array([target_spectral_moments(d) for d in train_data])
        yC_te = np.array([target_spectral_moments(d) for d in test_data])

        results = {}

        # ── Model A: energy regression ──
        print("\nTraining Model A (energy regression)...")
        modelA = make_mlp()
        modelA.fit(X_tr, yA_tr)
        predA = modelA.predict(X_te)
        errA = np.abs(predA - yA_te)
        results['model_A'] = {
            'label': 'Energy regression (baseline)',
            'pred': predA,
            'true': yA_te,
            'mae': errA.mean(),
            'per_point': errA,
        }
        print(f"  Test MAE (energy/site): {errA.mean():.4f} Ha")

        # ── Model B: cumulant regression (Idea I) ──
        print("\nTraining Model B (cumulant regression, Idea I)...")
        modelB = make_mlp(hidden=(128, 128))
        modelB.fit(X_tr, yB_tr)
        predB = modelB.predict(X_te)

        # Derive energies from predicted cumulants
        energies_B = []
        for i, d in enumerate(test_data):
            h, V = self._get_hV(d['U'])
            E = energy_from_prediction(
                predB[i], d['D1'], h, V, self.n_orb, mode='cumulant')
            energies_B.append(E / self.n_sites)
        energies_B = np.array(energies_B)
        errB_energy = np.abs(energies_B - yA_te)

        # Cumulant Frobenius error
        errB_cum = np.array([
            np.linalg.norm(predB[i] - yB_te[i]) for i in range(len(test_data))
        ])
        results['model_B'] = {
            'label': 'Cumulant regression, Idea I',
            'pred_energy': energies_B,
            'true_energy': yA_te,
            'mae_energy': errB_energy.mean(),
            'mae_cumulant': errB_cum.mean(),
            'per_point_energy': errB_energy,
            'per_point_cumulant': errB_cum,
        }
        print(f"  Test MAE (energy/site from cumulant): {errB_energy.mean():.4f} Ha")
        print(f"  Test MAE (cumulant Frobenius):        {errB_cum.mean():.4f}")

        # ── Model C: spectral moment regression (Idea II) ──
        print("\nTraining Model C (spectral moments, Idea II)...")
        modelC = make_mlp(hidden=(64, 64))
        modelC.fit(X_tr, yC_tr)
        predC = modelC.predict(X_te)
        errC = np.abs(predC - yC_te)
        results['model_C'] = {
            'label': 'Spectral moments, Idea II',
            'pred': predC,
            'true': yC_te,
            'mae_moments': errC.mean(),
            'per_point_moments': errC.mean(axis=1),
        }
        print(f"  Test MAE (spectral moments):          {errC.mean():.4f}")

        # ── Shot-budget experiment on Model B ──
        if n_shots_list is not None:
            print("\nShot-budget experiment (Model B with shadow noise)...")
            shot_results = {}
            rng = np.random.default_rng(0)
            for S in n_shots_list:
                noisy_cumsB = []
                for d in train_data:
                    D1_A, D1_B, D2_n = noisy_rdm(d['D1'], d['D2'], S, rng)
                    # Unbiased cumulant via sample splitting
                    mf = sample_split_wedge(D1_A, D1_B)
                    D1_avg = 0.5*(D1_A + D1_B)
                    cum_noisy = D2_n - mf
                    n = self.n_orb
                    Lmat = cum_noisy.reshape(n*n, n*n)
                    Lmat = 0.5*(Lmat + Lmat.T)
                    idx = np.triu_indices(n*n)
                    noisy_cumsB.append(Lmat[idx])

                yB_tr_noisy = np.array(noisy_cumsB)
                mB_noisy = make_mlp(hidden=(128, 128))
                mB_noisy.fit(X_tr, yB_tr_noisy)
                predB_noisy = mB_noisy.predict(X_te)

                errs = []
                for i, d in enumerate(test_data):
                    h, V = self._get_hV(d['U'])
                    E = energy_from_prediction(
                        predB_noisy[i], d['D1'], h, V, self.n_orb, mode='cumulant')
                    errs.append(abs(E / self.n_sites - yA_te[i]))
                shot_results[S] = np.array(errs)
                print(f"  S={S:7d} shots: MAE={np.mean(errs):.4f} Ha")

            results['shot_budget'] = shot_results

        results['U_test'] = np.array([d['U_over_t'] for d in test_data])
        results['U_train_range'] = (min(U_train)/self.t, max(U_train)/self.t)
        return results

    def run_invariance_check(self, U=4.0, n_rotations=10):
        """
        Verify that spectral moments are orbital-rotation invariant.
        Critical sanity check for Idea II.
        """
        print(f"\nInvariance check: {self.n_sites}-site Hubbard, U/t={U}")
        h, V, n_orb, n_elec = hubbard_chain(self.n_sites, self.t, U)
        E0, D1, D2, _ = exact_diag(h, V, n_orb, n_elec)
        moms0 = spectral_moments(D1, D2)

        rng = np.random.default_rng(42)
        max_dev = 0.0
        for _ in range(n_rotations):
            A = rng.standard_normal((n_orb, n_orb))
            Q, _ = np.linalg.qr(A)
            D1r = Q @ D1 @ Q.T
            D2r = np.einsum('pa,qb,abcd,rc,sd->pqrs', Q, Q, D2, Q, Q)
            moms_r = spectral_moments(D1r, D2r)
            dev = np.abs(moms_r - moms0).max()
            max_dev = max(max_dev, dev)

        print(f"  Max deviation in spectral moments over {n_rotations} rotations: {max_dev:.2e}")
        print(f"  {'PASS' if max_dev < 1e-8 else 'FAIL'} (threshold 1e-8)")
        return max_dev


# ── cross-system experiment (Idea II only) ────────────────────────────────────

def run_cross_system_experiment(n_train_sites, n_test_sites,
                                U_values, t=1.0, k_max=6):
    """
    Idea II exclusive: train on n_train_sites, evaluate on n_test_sites.
    Works because spectral moments are fixed-length regardless of system size.
    Model A and B cannot do this without architectural changes.
    """
    print(f"\n{'='*60}")
    print(f"Cross-system experiment (Idea II only)")
    print(f"  Train: {n_train_sites}-site Hubbard")
    print(f"  Test:  {n_test_sites}-site Hubbard")
    print(f"{'='*60}")

    train_data = generate_hubbard_dataset(n_train_sites, U_values, t)
    test_data  = generate_hubbard_dataset(n_test_sites,  U_values, t)

    X_tr = np.array([[d['U_over_t']] for d in train_data])
    X_te = np.array([[d['U_over_t']] for d in test_data])

    yC_tr = np.array([d['spectral_moments'][:k_max] for d in train_data])
    yC_te = np.array([d['spectral_moments'][:k_max] for d in test_data])

    yE_te = np.array([d['E0_per_site'] for d in test_data])  # ground truth

    model = make_mlp(hidden=(64, 64))
    model.fit(X_tr, yC_tr)
    pred = model.predict(X_te)

    err = np.abs(pred - yC_te).mean(axis=1)
    print(f"\n  Cross-system MAE in spectral moments: {err.mean():.4f}")
    print(f"  (No equivalent experiment possible for Model A or B)")

    return {
        'U_values': U_values,
        'pred_moments': pred,
        'true_moments': yC_te,
        'true_energies': yE_te,
        'mae_per_U': err,
    }

In [ ]:
"""
rdm_core.py
-----------
Exact-diagonalisation engine + cumulant machinery for the
two-month cumulant-ML project.

Extends the uploaded hubbard_dimer code to arbitrary Hubbard chains
and exposes the three objects needed by the ML pipeline:
    - D1  : 1-RDM
    - D2  : 2-RDM
    - cum : 2-body cumulant  (D2 - D1 ^ D1)

Also implements:
    - cumulant_spectrum   : eigenvalues of the matricized cumulant (Idea II)
    - spectral_moments    : tr(Lambda^k), the fixed-length invariant feature
    - sample_split_wedge  : unbiased wedge estimator (from the LaTeX proposal)
    - noisy_rdm           : simulates finite-shot shadow noise on D1, D2
"""

import numpy as np
from itertools import combinations


# ── fermionic second-quantisation helpers ────────────────────────────────────

def _below(det, p):
    """Number of occupied orbitals below position p in |det>."""
    return bin(det & ((1 << p) - 1)).count("1")

def _annihilate(p, det):
    if not (det >> p) & 1:
        return None
    return ((-1) ** _below(det, p), det & ~(1 << p))

def _create(p, det):
    if (det >> p) & 1:
        return None
    return ((-1) ** _below(det, p), det | (1 << p))

def _apply_ops(ops, state):
    """Apply a sequence of ('c'|'a', orbital) ops to a state dict {det: amp}."""
    for kind, orb in reversed(ops):
        fn = _create if kind == 'c' else _annihilate
        new = {}
        for det, amp in state.items():
            result = fn(orb, det)
            if result is not None:
                sign, new_det = result
                new[new_det] = new.get(new_det, 0.0) + sign * amp
        state = new
    return state

def _expectation(ops, psi):
    """<psi| ops |psi> where psi is a dict {det: amplitude}."""
    ket = _apply_ops(ops, psi)
    return sum(psi.get(det, 0.0) * amp for det, amp in ket.items())


# ── exact diagonalisation ────────────────────────────────────────────────────

def exact_diag(h, V, n_orb, n_elec):
    """
    Full CI for a system with n_orb spin-orbitals and n_elec electrons.

    Parameters
    ----------
    h : (n_orb, n_orb) one-body integrals
    V : (n_orb, n_orb, n_orb, n_orb) two-body integrals
        Convention: H2 = 0.5 * sum_{pqrs} V[p,q,r,s] a†_p a†_q a_s a_r
    n_orb, n_elec : int

    Returns
    -------
    E0  : float, ground-state energy
    D1  : (n_orb, n_orb) 1-RDM
    D2  : (n_orb, n_orb, n_orb, n_orb) 2-RDM
    psi : dict {det: amplitude}, ground-state wavefunction
    """
    basis = [sum(1 << i for i in occ)
             for occ in combinations(range(n_orb), n_elec)]
    pos = {det: i for i, det in enumerate(basis)}
    dim = len(basis)

    H = np.zeros((dim, dim))
    for j, det in enumerate(basis):
        ket = {det: 1.0}
        out = {}
        for p in range(n_orb):
            for q in range(n_orb):
                if h[p, q] != 0.0:
                    for d, a in _apply_ops([('c', p), ('a', q)], ket).items():
                        out[d] = out.get(d, 0.0) + h[p, q] * a
        for p in range(n_orb):
            for q in range(n_orb):
                for r in range(n_orb):
                    for s in range(n_orb):
                        if V[p, q, r, s] != 0.0:
                            for d, a in _apply_ops(
                                    [('c', p), ('c', q), ('a', s), ('a', r)], ket).items():
                                out[d] = out.get(d, 0.0) + 0.5 * V[p, q, r, s] * a
        for det2, amp in out.items():
            H[pos[det2], j] += amp

    w, vec = np.linalg.eigh(H)
    psi = {basis[i]: vec[i, 0] for i in range(dim)}

    r = n_orb
    D1 = np.array([[_expectation([('c', p), ('a', q)], psi)
                    for q in range(r)] for p in range(r)])
    D2 = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s in range(r):
                for t in range(r):
                    D2[p, q, s, t] = _expectation(
                        [('c', p), ('c', q), ('a', t), ('a', s)], psi)

    return w[0], D1, D2, psi


# ── Hubbard chain Hamiltonian ────────────────────────────────────────────────

def hubbard_chain(n_sites, t, U, periodic=False):
    """
    Fermi-Hubbard chain with n_sites sites, hopping t, interaction U.
    Spin-orbital index: 2*site + spin  (spin 0=up, 1=down).
    Half filling: n_elec = n_sites (one electron per site).

    Returns (h, V, n_orb, n_elec) ready for exact_diag.
    """
    r = 2 * n_sites
    h = np.zeros((r, r))
    V = np.zeros((r, r, r, r))

    # hopping (spin-preserving)
    for site in range(n_sites - 1):
        for spin in (0, 1):
            a, b = 2 * site + spin, 2 * (site + 1) + spin
            h[a, b] = h[b, a] = -t
    if periodic and n_sites > 2:
        for spin in (0, 1):
            a, b = spin, 2 * (n_sites - 1) + spin
            h[a, b] = h[b, a] = -t

    # on-site Hubbard U
    for site in range(n_sites):
        u, d = 2 * site, 2 * site + 1          # up / down spin-orbitals
        V[u, d, u, d] =  U / 2
        V[d, u, d, u] =  U / 2
        V[d, u, u, d] = -U / 2
        V[u, d, d, u] = -U / 2

    n_elec = n_sites    # half filling
    return h, V, r, n_elec


# ── cumulant and spectral features ───────────────────────────────────────────

def wedge(A, B):
    """
    Antisymmetrised outer product of two 1-RDMs.
    (A ^ B)_{pqrs} = 0.5*(A_{pr}B_{qs} - A_{ps}B_{qr} + B_{pr}A_{qs} - B_{ps}A_{qr})
    """
    return 0.5 * (
        np.einsum('pr,qs->pqrs', A, B) - np.einsum('ps,qr->pqrs', A, B)
        + np.einsum('pr,qs->pqrs', B, A) - np.einsum('ps,qr->pqrs', B, A)
    )

def cumulant(D1, D2):
    """2-body cumulant: Lambda = D2 - D1 ^ D1."""
    return D2 - wedge(D1, D1)

def sample_split_wedge(D1_A, D1_B):
    """
    Unbiased estimator of D1 ^ D1 using independent shot halves A, B.
    Removes the O(sigma^2) bias from the naive (D1_avg ^ D1_avg) estimator.
    See Section 4 of the LaTeX proposal.
    """
    return wedge(D1_A, D1_B)

def cumulant_matrix(D1, D2):
    """Matricize Lambda: (n^2 x n^2) real-symmetric matrix."""
    n = D1.shape[0]
    L = cumulant(D1, D2)
    Lmat = L.reshape(n * n, n * n)
    return 0.5 * (Lmat + Lmat.T)

def cumulant_spectrum(D1, D2):
    """
    Eigenvalues of the matricized cumulant (Idea II feature).
    Orbital-rotation invariant: if D1,D2 -> R D1 R^T, (R⊗R) D2 (R⊗R)^T
    then Lmat -> (R⊗R) Lmat (R⊗R)^T, spectrum unchanged.
    """
    return np.linalg.eigvalsh(cumulant_matrix(D1, D2))

def spectral_moments(D1, D2, k_max=6):
    """
    Fixed-length invariant feature vector: [tr(L), tr(L^2), ..., tr(L^k_max)].
    Basis- and system-size-agnostic. This is the Idea II ML feature.
    """
    eigs = cumulant_spectrum(D1, D2)
    return np.array([np.sum(eigs ** k) for k in range(1, k_max + 1)])

def correlation_measure(D1):
    """
    Scalar correlation strength: tr[D1(I - D1)].
    Zero at HF, grows with correlation. Equals -sum_k lambda_k (cumulant trace).
    """
    return np.trace(D1 @ (np.eye(len(D1)) - D1))


# ── simulated shadow noise ────────────────────────────────────────────────────

def noisy_rdm(D1_exact, D2_exact, n_shots, rng=None):
    """
    Simulate finite-shot Pauli shadow noise on D1 and D2.

    Models each RDM element as having Gaussian noise with variance
    proportional to 1/n_shots (valid in the large-shot limit for
    classical shadows). Returns noisy D1_A, D1_B (independent halves)
    and noisy D2, enabling the sample-split-wedge debiasing.

    This is a *statistical model* of shadow noise, not a full
    shadow-tomography simulation. Replace with a real shadow
    implementation when moving to hardware.
    """
    if rng is None:
        rng = np.random.default_rng()

    # Pauli shadow variance ~ O(1/shots) for weight-2 (D1) and weight-4 (D2) terms
    sigma1 = 1.0 / np.sqrt(n_shots)          # 1-body noise scale
    sigma2 = 4.0 / np.sqrt(n_shots)          # 2-body noise scale (larger weight)

    n = D1_exact.shape[0]
    shots_A = n_shots // 2
    shots_B = n_shots - shots_A

    # Independent noise for each half (needed for unbiased wedge)
    noise_A = rng.standard_normal((n, n)) * (1.0 / np.sqrt(shots_A))
    noise_B = rng.standard_normal((n, n)) * (1.0 / np.sqrt(shots_B))
    D1_A = D1_exact + sigma1 * noise_A
    D1_B = D1_exact + sigma1 * noise_B

    noise_D2 = rng.standard_normal((n, n, n, n)) * sigma2
    # Enforce antisymmetry of the noise
    noise_D2 = 0.25 * (noise_D2 - noise_D2.transpose(1,0,2,3)
                        - noise_D2.transpose(0,1,3,2) + noise_D2.transpose(1,0,3,2))
    D2_noisy = D2_exact + noise_D2

    return D1_A, D1_B, D2_noisy


# ── dataset generation ────────────────────────────────────────────────────────

def generate_hubbard_dataset(n_sites, U_values, t=1.0, verbose=True):
    """
    Generate a dataset of (U, E0, D1, D2, cumulant, spectrum, moments)
    for a Hubbard chain at half filling across a range of U/t values.

    Returns a list of dicts, one per U value.
    """
    dataset = []
    for U in U_values:
        h, V, n_orb, n_elec = hubbard_chain(n_sites, t, U)
        E0, D1, D2, _ = exact_diag(h, V, n_orb, n_elec)
        cum = cumulant(D1, D2)
        eigs = cumulant_spectrum(D1, D2)
        moms = spectral_moments(D1, D2)
        corr = correlation_measure(D1)
        dataset.append({
            'U': U, 't': t, 'U_over_t': U / t,
            'n_sites': n_sites, 'n_orb': n_orb,
            'E0': E0, 'E0_per_site': E0 / n_sites,
            'D1': D1, 'D2': D2,
            'cumulant': cum,
            'cumulant_eigs': eigs,
            'spectral_moments': moms,
            'correlation_measure': corr,
        })
        if verbose:
            print(f"  n={n_sites}, U/t={U/t:5.1f} | "
                  f"E0/site={E0/n_sites:+.4f} | "
                  f"corr={corr:.4f} | "
                  f"max|lambda|={np.abs(eigs).max():.4f}")
    return dataset

### Newest code version:

In [ ]:
"""
spectral_cumulant.py
====================================================================
Starter code for "Idea II": cumulant-spectrum invariants as transferable,
basis-free descriptors of electron correlation for machine learning.

Core object: the matricized 2-cumulant  L[p,q,r,s] -> Lmat[(pq),(rs)].
Lmat is real-symmetric, so its eigenvalues are real. Under any orbital
rotation R,  L -> (R(x)R) L (R(x)R)^T, i.e. Lmat is conjugated by a unitary,
so its SPECTRUM is invariant. The spectrum (and the moments tr Lmat^k) is
therefore a gauge-free, fixed-length descriptor of correlation.

Exact constraints used as runtime checks:
    sum_k lambda_k = tr Lmat = -tr[D1 (I - D1)]   (fixed by the 1-RDM)
    lambda_k supported on antisym 2-particle subspace (rank <= r(r-1)/2)

What this file gives you:
    - exact ED for small Hubbard chains (ground-truth 1-/2-RDM)
    - cumulant, its spectrum, spectral moments, and the exact checks
    - a dataset generator over (system size, U/t)
    - the three starter experiments with HONEST baselines:
        (A) gauge invariance   -> should pass to ~1e-15
        (B) size-transfer       -> spectral features vs scalar/extensive baselines
        (C) Weyl noise stability-> spectrum drift bounded by perturbation size
    - clearly marked TODOs for the next phase (heterogeneous systems,
      shadow-noise labels, better intensive normalization, real models)

Dependencies: numpy, scikit-learn (only for the toy regressors).
Author: starter scaffold — extend freely.
"""

from __future__ import annotations
import numpy as np
from itertools import combinations

# =====================================================================
# 1.  Minimal fermionic exact diagonalization (ground-truth RDMs)
#     Occupation-number (bitstring) Fock space; Jordan-Wigner phases.
# =====================================================================

def _below(d, p):            # parity of occupied orbitals below p
    return bin(d & ((1 << p) - 1)).count("1")

def _ann(p, d):
    return None if not (d >> p) & 1 else ((-1) ** _below(d, p), d & ~(1 << p))

def _cre(p, d):
    return None if (d >> p) & 1 else ((-1) ** _below(d, p), d | (1 << p))

def _apply(ops, state):
    """Apply a string of ('c'|'a', orbital) operators (rightmost first)."""
    for kind, orb in reversed(ops):
        new, f = {}, (_cre if kind == "c" else _ann)
        for det, amp in state.items():
            r = f(orb, det)
            if r:
                sign, nd = r
                new[nd] = new.get(nd, 0.0) + sign * amp
        state = new
    return state

def _exp(ops, psi):
    ket = _apply(ops, psi)
    return sum(psi.get(d, 0.0) * a for d, a in ket.items())

def ground_rdms(h, V, r, N):
    """
    Exact ground state of  H = sum h_pq a_p^ a_q + 1/2 sum V_pqst a_p^ a_q^ a_t a_s
    in the N-particle sector of r spin-orbitals.
    Returns (E0, D1, D2) with
        D1[p,q]     = <a_p^ a_q>
        D2[p,q,s,t] = <a_p^ a_q^ a_t a_s>
    """
    basis = [sum(1 << i for i in occ) for occ in combinations(range(r), N)]
    pos = {d: i for i, d in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for j, det in enumerate(basis):
        ket, out = {det: 1.0}, {}
        for p in range(r):
            for q in range(r):
                if h[p, q]:
                    for d, a in _apply([("c", p), ("a", q)], ket).items():
                        out[d] = out.get(d, 0.0) + h[p, q] * a
        for p in range(r):
            for q in range(r):
                for s in range(r):
                    for t in range(r):
                        if V[p, q, s, t]:
                            for d, a in _apply([("c", p), ("c", q),
                                                ("a", t), ("a", s)], ket).items():
                                out[d] = out.get(d, 0.0) + 0.5 * V[p, q, s, t] * a
        for d, a in out.items():
            H[pos[d], j] += a
    w, vec = np.linalg.eigh(H)
    psi = {basis[i]: vec[i, 0] for i in range(len(basis))}
    D1 = np.array([[_exp([("c", p), ("a", q)], psi) for q in range(r)]
                   for p in range(r)])
    D2 = np.zeros((r, r, r, r))
    for p in range(r):
        for q in range(r):
            for s in range(r):
                for t in range(r):
                    D2[p, q, s, t] = _exp([("c", p), ("c", q),
                                           ("a", t), ("a", s)], psi)
    return w[0], D1, D2


# =====================================================================
# 2.  Model systems
# =====================================================================

def hubbard_chain(L, t, U, pbc=False):
    """L sites at half filling (N=L electrons). Spin-orbital = 2*site + spin."""
    r = 2 * L
    h = np.zeros((r, r))
    bonds = [(i, i + 1) for i in range(L - 1)]
    if pbc and L > 2:
        bonds.append((L - 1, 0))
    for (i, j) in bonds:
        for spin in (0, 1):
            a, b = 2 * i + spin, 2 * j + spin
            h[a, b] += -t
            h[b, a] += -t
    V = np.zeros((r, r, r, r))
    for site in range(L):
        u, d = 2 * site + 0, 2 * site + 1
        V[u, d, u, d] = V[d, u, d, u] = U / 2
        V[d, u, u, d] = V[u, d, d, u] = -U / 2
    return h, V, r, L

# TODO (Phase 2): heterogeneous systems where the spectral SHAPE varies
# independently of its trace -- e.g. two coupled dimers with different U,
# an asymmetric 3-site cluster, or a minimal molecular active space from
# PySCF. Hubbard's site-homogeneity makes the spectrum nearly single-scale,
# which is exactly why higher moments add little there (see experiment B).


# =====================================================================
# 3.  Cumulant, spectrum, invariant features, and exact checks
# =====================================================================

def wedge(A, B):
    return 0.5 * (np.einsum("pr,qs->pqrs", A, B) - np.einsum("ps,qr->pqrs", A, B)
                  + np.einsum("pr,qs->pqrs", B, A) - np.einsum("ps,qr->pqrs", B, A))

def cumulant(D1, D2):
    return D2 - wedge(D1, D1)

def cumulant_matrix(D1, D2):
    r = D1.shape[0]
    Lm = cumulant(D1, D2).reshape(r * r, r * r)
    return 0.5 * (Lm + Lm.T)                 # enforce exact symmetry

def cumulant_spectrum(D1, D2):
    return np.linalg.eigvalsh(cumulant_matrix(D1, D2))

def spectral_moments(eigs, kmax=4):
    """tr(Lmat^k) = sum lambda^k. Fixed length, basis- and size-agnostic."""
    return np.array([np.sum(eigs ** k) for k in range(1, kmax + 1)])

def check_constraints(D1, D2, tol=1e-8):
    """Verify the two exact cumulant identities; returns a dict of residuals."""
    r = D1.shape[0]
    eigs = cumulant_spectrum(D1, D2)
    trace_id = abs(eigs.sum() - (-np.trace(D1 @ (np.eye(r) - D1))))
    rank_ok = (np.sum(np.abs(eigs) > 1e-9) <= r * (r - 1) // 2)
    return dict(trace_residual=trace_id, rank_ok=bool(rank_ok))

def rotate_rdms(D1, D2, R):
    """Orbital rotation R in O(r)/U(r); spectrum must be invariant under this."""
    return R @ D1 @ R.T, np.einsum("pa,qb,abcd,rc,sd->pqrs", R, R, D2, R, R)


# =====================================================================
# 4.  Reference energies and dataset generation
# =====================================================================

def hf_energy(h, V, r, N):
    """Mean-field (idempotent-D1) reference energy = E with cumulant set to 0."""
    _, vec = np.linalg.eigh(h)
    occ = vec[:, :N]
    D1 = occ @ occ.T
    return (np.einsum("pq,pq->", h, D1)
            + 0.5 * np.einsum("pqrs,pqrs->", V, wedge(D1, D1)))

def make_dataset(L, Us, t=1.0, kmax=4):
    rows = []
    for U in Us:
        h, V, r, N = hubbard_chain(L, t, U)
        E0, D1, D2 = ground_rdms(h, V, r, N)
        eigs = cumulant_spectrum(D1, D2)
        rows.append(dict(
            L=L, U=float(U), N=N, r=r, E0=E0,
            Ehf=hf_energy(h, V, r, N), Ecorr=E0 - hf_energy(h, V, r, N),
            eigs=eigs,
            moments=spectral_moments(eigs, kmax),
            moments_per_e=spectral_moments(eigs, kmax) / N,
        ))
    return rows


# =====================================================================
# 5.  Starter experiments (run as __main__)
# =====================================================================

def _exp_gauge():
    print("[A] GAUGE INVARIANCE")
    h, V, r, N = hubbard_chain(4, 1.0, 4.0)
    _, D1, D2 = ground_rdms(h, V, r, N)
    e0 = cumulant_spectrum(D1, D2)
    rng = np.random.default_rng(1)
    dev = 0.0
    for _ in range(6):
        Q, _ = np.linalg.qr(rng.standard_normal((r, r)))
        D1r, D2r = rotate_rdms(D1, D2, Q)
        dev = max(dev, np.abs(np.sort(cumulant_spectrum(D1r, D2r)) - np.sort(e0)).max())
    chk = check_constraints(D1, D2)
    print(f"    max spectrum drift over 6 random rotations : {dev:.2e}  (expect ~1e-15)")
    print(f"    trace-identity residual                    : {chk['trace_residual']:.2e}")
    print(f"    rank bound satisfied                       : {chk['rank_ok']}\n")

def _exp_transfer():
    from sklearn.kernel_ridge import KernelRidge
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    print("[B] SIZE TRANSFER  (leave-one-size-out; target = E_corr per electron)")
    Us = np.linspace(0.0, 8.0, 33)
    DS = {L: make_dataset(L, Us) for L in (2, 3, 4)}
    def feats(ds, kind):
        if kind == "intensive": return np.array([d["moments_per_e"] for d in ds])
        if kind == "extensive": return np.array([d["moments"] for d in ds])
        if kind == "scalar":    return np.array([[d["moments_per_e"][0]] for d in ds])
    def tgt(ds): return np.array([d["Ecorr"] / d["N"] for d in ds])
    print(f"    {'test L':>6} | {'intensive moments':>17} | "
          f"{'extensive moments':>17} | {'scalar (trL/N)':>15}")
    for testL in (2, 3, 4):
        trains = [L for L in (2, 3, 4) if L != testL]
        out = {}
        for kind in ("intensive", "extensive", "scalar"):
            Xtr = np.vstack([feats(DS[L], kind) for L in trains])
            ytr = np.concatenate([tgt(DS[L]) for L in trains])
            Xte, yte = feats(DS[testL], kind), tgt(DS[testL])
            m = make_pipeline(StandardScaler(),
                              KernelRidge(alpha=1e-5, kernel="rbf", gamma=0.3))
            m.fit(Xtr, ytr)
            out[kind] = np.mean(np.abs(m.predict(Xte) - yte))
        print(f"    {testL:>6} | {out['intensive']:>17.4e} | "
              f"{out['extensive']:>17.4e} | {out['scalar']:>15.4e}")
    print("    NOTE: on homogeneous Hubbard the scalar trace baseline is strong;")
    print("    higher moments do not yet beat it. Phase 2 tests heterogeneous")
    print("    systems where the spectral SHAPE varies independently of its trace.\n")

def _exp_noise():
    print("[C] WEYL NOISE STABILITY  (L=4, U=4)")
    h, V, r, N = hubbard_chain(4, 1.0, 4.0)
    _, D1, D2 = ground_rdms(h, V, r, N)
    eigs0 = cumulant_spectrum(D1, D2)
    rng = np.random.default_rng(0)
    print(f"    {'sigma':>8} | {'max|d lambda|':>13} | {'op-norm ||E||':>13}  (Weyl: left<=right)")
    for sigma in (1e-3, 3e-3, 1e-2, 3e-2):
        md, mn = [], []
        for _ in range(40):
            noise = rng.normal(0, sigma, D2.shape)
            D2n = D2 + 0.5 * (noise + noise.transpose(1, 0, 3, 2))
            E = cumulant_matrix(D1, D2n) - cumulant_matrix(D1, D2)
            md.append(np.abs(np.sort(cumulant_spectrum(D1, D2n)) - np.sort(eigs0)).max())
            mn.append(np.linalg.norm(E, 2))
        print(f"    {sigma:8.0e} | {np.mean(md):13.3e} | {np.mean(mn):13.3e}")
    print("    -> eigenvalue drift is bounded by the perturbation's operator norm,")
    print("    so spectral features degrade gracefully (and predictably) with shots.\n")

if __name__ == "__main__":
    np.set_printoptions(precision=4, suppress=True)
    print("=" * 68)
    print(" spectral cumulant features — starter experiments")
    print("=" * 68 + "\n")
    _exp_gauge()
    _exp_transfer()
    _exp_noise()
    print("Done. See TODOs in the source for the next phase.")

# Idea II — Work plan for the coming weeks

The starter code (`spectral_cumulant.py`) already gives you exact ground truth,
the invariant features, and three working experiments. Here's the sequence to
turn it into a result, with honest decision gates so you don't sink time into a
dead branch.

## The one finding that shapes everything

On homogeneous Hubbard, the scalar `tr Λ / N` (= non-idempotency of the 1-RDM,
moment 1) is already a strong predictor and the higher moments don't beat it.
That's not a failure — it's a diagnosis. Hubbard's site-homogeneity makes the
correlation spectrum nearly single-scale, so moments 2,3,4 are near-functions of
moment 1. The whole project hinges on showing the spectral *shape* carries
*independent* transferable signal, which means **your first job is to build
systems where the shape decouples from the trace.** If it never decouples, the
honest paper is "spectrum ≈ trace for these systems"; if it does, you have the
result.

## Weeks 1–2 — Heterogeneous benchmark + information gate

- Add 2–3 heterogeneous systems to the `# TODO (Phase 2)` hook in the starter:
  - two 2-site Hubbard dimers with *different* U coupled by a weak hop
  - an asymmetric 3-site cluster (site-dependent U or on-site energies)
  - one minimal molecular active space from PySCF (e.g. H₂ + H₂ at different
    separations, or a (4e,4o) CAS) to leave the lattice world
- For each system, compute the **conditional information of (μ₂,μ₃,μ₄) given μ₁**:
  regress μ₂…μ₄ on μ₁; if the residuals are large and structured, the shape is
  independent and the project is alive. This is a cheap, decisive gate.

**Decision gate:** if no system shows independent higher-moment signal, stop and
reframe before building ML. If several do, proceed.

## Weeks 3–4 — Principled intensive features + transfer

- Replace the crude `moments_per_e` with size-consistent normalisations and
  compare them head-to-head on leave-one-size-out transfer:
  - eigenvalue density / histogram features (binned spectrum)
  - moments per correlated mode (divide by `rank`, not by N)
  - log-moments or cumulants-of-the-spectrum (often more Gaussian, better for KRR)
- Move from kernel ridge to a **Gaussian process** (scikit-learn `GaussianProcessRegressor`)
  so every prediction carries an uncertainty — essential for the
  out-of-distribution transfer claim and cheap at this data scale.

**Deliverable:** a transfer table (and parity plots with error bars) showing
which intensive spectral representation transports best, and whether it beats the
scalar baseline on the *heterogeneous* set.

## Weeks 5–6 — Label from the pipeline, not from ED

This is what ties Idea II back to Idea I and makes it a quantum result rather than
a classical curiosity.

- Wrap the Idea I estimator (start with simulated Pauli/fermionic shadows +
  sample-split debiasing) so you can produce **noisy** spectra at a chosen shot
  budget instead of exact ED spectra.
- Retrain the best model from weeks 3–4 on shadow-noisy spectra. Plot energy/
  property error vs shot budget and check it against the Weyl prediction
  (`max|Δλ| ≤ ‖E‖₂`, already verified in experiment C).

**Deliverable:** an accuracy-vs-shots curve showing the spectral features stay
useful well below the shot count needed to resolve individual cumulant entries.
That is the practical payoff — fewer shots because you only need the spectrum.

## Weeks 7–8 — Consolidate and stress-test

- Leave-one-*chemistry*-out (train on lattices, test on a molecule, or vice
  versa) — the hardest and most convincing transfer test.
- Robustness sweeps: how does the gate-1 conclusion hold as you vary system
  heterogeneity, filling, and basis? Where does the spectral picture break?
- Start the methods write-up using the figures you've accumulated. The amended
  proposal section (`idea_II.tex`) is the skeleton of the methods + theory.

## Standing rules to keep it honest

- Always run `check_constraints` on any reconstructed RDM before trusting a
  spectrum (trace identity + rank bound catch most bugs instantly).
- Report the scalar `tr Λ / N` baseline on *every* experiment. The paper's claim
  is only ever "shape beats trace" where that's actually true.
- Keep ED ground truth alongside every shadow-noisy result so error is always
  decomposable into (measurement noise) + (model error).

## What success looks like at ~8 weeks

A defensible claim with evidence: *the gauge-invariant cumulant spectrum is a
Weyl-stable, transferable descriptor of electron correlation; on heterogeneous
systems its shape carries information beyond the scalar non-idempotency, and that
information survives shadow-level measurement noise.* That is the core of a paper
and the foundation for the larger 4–6 month program (heterogeneous chemistry at
scale, real hardware spectra, regime classification).